# Train a ConvLSTM on June 2026 — full-res, all 16 bands, same-day grid

From a CST day's 24 hourly GOES frames (all 16 bands, full 1500×2500), predict that day's NWS flood-warning grid (50 km CONUS-land cells). Self-contained ConvLSTM (Encoder → ConvLSTM → CellPool → head, defined in `convlstm_june_train.py`); **images-only** head by default.

Inputs: GOES-19 hourly `/mnt/disk4/recent-goes`. Outputs: `signatures/data/flood_warnings_2026.parquet` → grid. Training runs on both GPUs via `torchrun convlstm_june_train.py`.

## 0 · Config, paths & constants

In [ ]:
import json, sys, time
from datetime import datetime, timedelta
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd
import shapely.wkb

ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "signatures" / "notebooks"))  # for `import convlstm_june_train`
from config import build_grid_cells, grid_transform, CELL_KM, IMG_H, IMG_W  # noqa: E402

# ---- paths (June "signatures" work lives under signatures/) ----
GOES_DIR = Path("/mnt/disk4/recent-goes")                  # hourly GOES-19 (not moved)
WARN_PARQUET = ROOT / "signatures/data/flood_warnings_2026.parquet"
CACHE_DIR = ROOT / "signatures/cache/convlstm_june"        # on the project drive
CKPT_DIR = ROOT / "signatures/notebooks"                   # trained model lives by the nb
STATS_PATH = CACHE_DIR / "band_stats_2026.json"
CACHE_DIR.mkdir(parents=True, exist_ok=True); CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ---- problem constants ----
ALL_BANDS = list(range(1, 17))            # all 16 ABI bands, raw (no band selection)
N_BAND = len(ALL_BANDS)                   # 16
USE_TIME = True                           # append a per-frame CST-hour channel
N_CH = N_BAND + (1 if USE_TIME else 0)    # 17 model input channels per frame
T_TARGET = 24                             # hourly frames per CST day
CST_OFFSET = timedelta(hours=6)           # CST = UTC - 6 (fixed; CST midnight = 06:00 UTC)

from config import TVERSKY_ALPHA, TVERSKY_BETA  # noqa: E402
EPOCHS = 100
LR = 3e-4
plt.rcParams.update({"figure.dpi": 110})
print(f"input {T_TARGET} CST-hourly frames x {N_CH} ch x {IMG_H}x{IMG_W}  ->  {CELL_KM} km grid")
print(f"per-sample x  ~{T_TARGET*N_BAND*IMG_H*IMG_W*2/1e9:.2f} GB f16   cache -> {CACHE_DIR}")

## 1 · Output grid + full-res GOES-pixel → cell index

In [ ]:
from convlstm_june_train import pool_sub                  # self-contained model utils

cells, GRID_R, GRID_C, land_mask = build_grid_cells()
print(f"{CELL_KM} km grid ({GRID_R}, {GRID_C}); land cells {int(land_mask.sum())}")

def build_pix2cell_recent():
    import pyproj
    cache = CACHE_DIR / f"pix2cell_{CELL_KM}km.npy"
    if cache.exists():
        return np.load(cache)
    gx0, gy0, gstep, *_ = grid_transform()
    ref = sorted(GOES_DIR.glob("*/2026/*/*/*.nc"))[0]
    with netCDF4.Dataset(ref) as nc:
        proj = nc["goes_imager_projection"]
        geos = pyproj.CRS.from_cf({k: proj.getncattr(k) for k in proj.ncattrs()})
        sat_h = float(proj.perspective_point_height)
        xc = nc["x"][:].astype(np.float64) * sat_h
        yc = nc["y"][:].astype(np.float64) * sat_h
    xx, yy = np.meshgrid(xc, yc)
    tf = pyproj.Transformer.from_crs(geos, 5070, always_xy=True)
    ax, ay = tf.transform(xx.ravel(), yy.ravel())
    ax = np.asarray(ax).reshape(xx.shape); ay = np.asarray(ay).reshape(xx.shape)
    col = np.floor((ax - gx0) / gstep)
    row = (GRID_R - 1) - np.floor((ay - gy0) / gstep)
    ok = (np.isfinite(ax) & np.isfinite(ay)
          & (col >= 0) & (col < GRID_C) & (row >= 0) & (row < GRID_R))
    p2c = np.full(ax.shape, -1, np.int32)
    p2c[ok] = (row[ok] * GRID_C + col[ok]).astype(np.int32)
    np.save(cache, p2c)
    return p2c

PIX2CELL = build_pix2cell_recent()
SUB = pool_sub(PIX2CELL, stride=2)                         # encoder-grid index for CellPool
print("pix2cell", PIX2CELL.shape, "-> sub", SUB.shape,
      f"({(PIX2CELL >= 0).mean()*100:.0f}% of pixels on grid)")

## 2 · Group the hourly UTC frames into CST local days (hours 0–23)

In [ ]:
def scan_token(p):
    for part in p.name.split("_"):
        if part.startswith("s") and part[1:].isdigit():
            return part
    return p.name

def scan_utc(p):
    return datetime.strptime(scan_token(p)[1:12], "%Y%j%H%M")   # UTC scan-start

files = sorted(GOES_DIR.glob("*/2026/06/*/*.nc"), key=scan_token)

from collections import defaultdict
day_frames = defaultdict(list)
for p in files:
    cst = scan_utc(p) - CST_OFFSET
    day_frames[cst.date()].append((cst.hour, p))
for d in day_frames:
    day_frames[d] = sorted(day_frames[d])                  # by CST hour

counts = {d: len(v) for d, v in sorted(day_frames.items())}
print("frames per CST day:", {str(d): n for d, n in counts.items()})

SAMPLE_DAYS = [d for d in sorted(day_frames) if len(day_frames[d]) == T_TARGET]
print(f"\ncomplete (24-frame) CST days: {len(SAMPLE_DAYS)}  "
      f"{SAMPLE_DAYS[0]} .. {SAMPLE_DAYS[-1]}")

## 3 · Labels — warnings rasterized to the grid, by CST issue day

In [ ]:
w = pd.read_parquet(WARN_PARQUET)
w["geometry"] = w["geometry"].apply(shapely.wkb.loads)
w = gpd.GeoDataFrame(w, geometry="geometry", crs=4326)
w["cst_day"] = (w["issue_date"] - CST_OFFSET).dt.date

labels = {}
for d in SAMPLE_DAYS:
    wd = w[w["cst_day"] == d]
    a = np.zeros((GRID_R, GRID_C), np.float32)
    if len(wd):
        j = gpd.sjoin(cells, wd[["geometry"]], predicate="intersects", how="inner")
        a[cells.loc[j.index.unique(), "R"], cells.loc[j.index.unique(), "C"]] = 1.0
    labels[d] = a

pos = np.array([labels[d][land_mask].sum() for d in SAMPLE_DAYS])
base = pos.sum() / (len(SAMPLE_DAYS) * land_mask.sum())
print(f"flooded land cells/day: median {np.median(pos):.0f}, max {pos.max():.0f}  "
      f"(base rate {base*100:.2f}%)")

## 4 · Per-band normalization (all 16 bands)

In [ ]:
if STATS_PATH.exists():
    stats = json.loads(STATS_PATH.read_text())
    print("loaded cached band stats")
else:
    rng = np.random.default_rng(0)
    sample_files = [day_frames[d][h][1] for d in SAMPLE_DAYS[:6] for h in (8, 13, 18, 22)]
    acc = {b: [] for b in ALL_BANDS}
    for f in sample_files:
        with netCDF4.Dataset(f) as nc:
            for b in ALL_BANDS:
                a = np.ma.filled(nc[f"CMI_C{b:02d}"][:].astype(np.float32), np.nan)
                acc[b].append(a[::4, ::4])
    stats = {}
    for b in ALL_BANDS:
        s = np.stack(acc[b])
        stats[str(b)] = {"mean": float(np.nanmean(s)), "std": float(np.nanstd(s) or 1.0)}
    STATS_PATH.write_text(json.dumps(stats, indent=2))
    print(f"computed band stats -> {STATS_PATH.name}")

BAND_MEAN = np.array([stats[str(b)]["mean"] for b in ALL_BANDS], np.float32)
BAND_STD = np.array([stats[str(b)]["std"] for b in ALL_BANDS], np.float32)
for b, m, s in zip(ALL_BANDS, BAND_MEAN, BAND_STD):
    print(f"  band {b:>2}: mean {m:8.3f}  std {s:7.3f}")

## 5 · Materialize the cache (full-res, 16 bands, 24 CST frames)

In [ ]:
def load_sample(day):
    frames = day_frames[day]                               # [(cst_hour, path), ...] sorted
    x = np.zeros((len(frames), N_BAND, IMG_H, IMG_W), np.float16)
    for t, (_, f) in enumerate(frames):
        with netCDF4.Dataset(f) as nc:
            for c, b in enumerate(ALL_BANDS):
                a = np.ma.filled(nc[f"CMI_C{b:02d}"][:].astype(np.float32), np.nan)
                a = (a - BAND_MEAN[c]) / BAND_STD[c]
                x[t, c] = np.nan_to_num(a).astype(np.float16)
    hours = np.array([h for h, _ in frames], np.float32)   # CST hour of each frame (0..23)
    return x, hours, labels[day]

N_TEST = 5
SPLIT_SEED = 420
n = len(SAMPLE_DAYS)
test_idx = np.random.default_rng(SPLIT_SEED).choice(n, size=N_TEST, replace=False)
split = np.array(["train"] * n, dtype=object)
split[test_idx] = "test"
manifest = pd.DataFrame({"cst_day": [pd.Timestamp(d) for d in SAMPLE_DAYS],
                         "n_pos": pos.astype(int), "split": split})
manifest.to_parquet(CACHE_DIR / "manifest.parquet")
print(manifest["split"].value_counts().reindex(["train", "val", "test"]).to_string())
print(f"train {int((split=='train').sum())}  val 0  test {N_TEST}  (random, seed {SPLIT_SEED})  "
      f"test days: {sorted(SAMPLE_DAYS[i].strftime('%m-%d') for i in test_idx)}")

from concurrent.futures import ProcessPoolExecutor
BUILD_CACHE = True
N_WORKERS = 6

def _write_one(day):
    stem = CACHE_DIR / day.strftime("%Y%m%d")
    if Path(f"{stem}_x.npy").exists():
        return 0
    x, hours, y = load_sample(day)
    np.save(f"{stem}_x.npy", x)
    np.save(f"{stem}_t.npy", hours)
    np.save(f"{stem}_y.npy", y.astype(np.uint8))
    return x.nbytes

if BUILD_CACHE:
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        nbytes = list(ex.map(_write_one, SAMPLE_DAYS))
    print(f"cache built: {sum(nbytes)/1e9:.1f} GB in {(time.perf_counter()-t0)/60:.1f} min")
else:
    print("BUILD_CACHE=False (using existing cache)")

## 6 · Model & data — defined in `convlstm_june_train.py`

The dataset + checkpointed ConvLSTM live in the script (shared with the torchrun workers); imported here only to size the model.

In [ ]:
import torch  # noqa: F401
from convlstm_june_train import SameDayCache, ConvLSTMNetCkpt, build_sub, splits

sub, gr, gc, land = build_sub()                       # 50 km grid + pixel->cell index
tr_days, va_days, te_days = splits()                  # from manifest.parquet

net_cpu = ConvLSTMNetCkpt(sub, gr, gc, cin=N_CH)   # images-only; CPU param count
print(f"params {sum(p.numel() for p in net_cpu.parameters()):,}  (images-only head)  | "
      f"train {len(tr_days)} val {len(va_days)} test {len(te_days)}")
del net_cpu

## 7 · Train on both GPUs via torchrun

Launches `convlstm_june_train.py` with `torchrun --nproc_per_node=2` (fork-based DDP from a Jupyter kernel deadlocks). `BATCH_PER_GPU` sets VRAM (3 → ~76 GB).

Set `USE_GLM = True` below to also feed GLM lightning (binned to the cell grid, concatenated before the head).

In [ ]:
import os, subprocess

BATCH_PER_GPU = 3      # per GPU; VRAM ~ 1->26 / 2->51 / 3->76 GB of 94 (RAM stays bounded)
USE_GLM = False        # True -> also feed GLM lightning (per-cell count/energy/area)
RESULTS_NPZ = CACHE_DIR / "ddp_results.npz"

cmd = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2",
       "convlstm_june_train.py", "--epochs", str(EPOCHS), "--batch", str(BATCH_PER_GPU),
       "--workers", "0"]
if USE_GLM:
    cmd.append("--glm")
env = {**os.environ, "NCCL_P2P_DISABLE": "1",                # required on this box
       "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
print("launching:", " ".join(cmd[1:]), flush=True)
proc = subprocess.Popen(cmd, cwd=str(ROOT / "signatures/notebooks"), env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:                                     # live training log
    if any(k in line for k in ("[ddp]", "TEST", "Error", "Traceback", "Killed")):
        print(line.rstrip())
proc.wait()
print("torchrun exit", proc.returncode)
assert proc.returncode == 0, "training failed — see the streamed log above"

## 8 · Evaluate + maps

No validation split (train = all non-test days), so the trainer monitors the **test set** each epoch — progress tracking, not an independent estimate.

In [ ]:
from sklearn.metrics import average_precision_score
r = np.load(RESULTS_NPZ, allow_pickle=True)
hist = r["hist"]
P, Y, te_days_s = r["probs"], r["trues"], r["te_days"]
vP, vY, va_days_s = r["val_probs"], r["val_trues"], r["va_days"]
base_te = Y[:, land_mask].mean()
test_ap = average_precision_score(Y[:, land_mask].ravel(), P[:, land_mask].ravel())
print(f"TEST PR-AUC {test_ap:.4f}  (base {base_te*100:.2f}%  ->  {test_ap/base_te:.1f}x)  "
      f"checkpoint: {CKPT_DIR/'convlstm_june.pt'}")

ep, trl, test_curve = hist.T
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(ep, trl, "o-"); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(ep, test_curve, "o-"); ax[1].axhline(base_te, color="k", ls="--", lw=0.8)
ax[1].set_title("test PR-AUC (per epoch)"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

Below, one row per day: **predicted P(flood)** (viridis) next to the **truth** (red =
warned cells). Same-day GOES → same-day warning grid, on the 50 km CONUS-land cells.

In [ ]:
bg = np.where(land_mask, 0.12, np.nan)            # grey land, NaN ocean

def map_grid(Pset, Yset, days, title):
    n = len(days)
    fig, axes = plt.subplots(n, 2, figsize=(11, 3.0 * n), squeeze=False)
    for i, d in enumerate(days):
        d = str(d)
        ap = (average_precision_score(Yset[i][land_mask], Pset[i][land_mask])
              if Yset[i][land_mask].sum() else float("nan"))
        a0, a1 = axes[i]
        a0.imshow(bg, cmap="Greys", vmin=0, vmax=1)
        im = a0.imshow(np.where(land_mask, Pset[i], np.nan), cmap="viridis", vmin=0, vmax=1)
        a0.set_title(f"{d}  predicted P(flood)   PR-AUC {ap:.3f}", fontsize=9)
        a1.imshow(bg, cmap="Greys", vmin=0, vmax=1)
        a1.imshow(np.where(land_mask, Yset[i], np.nan), cmap="Reds", vmin=0, vmax=1)
        a1.set_title(f"{d}  truth  ({int(Yset[i][land_mask].sum())} warned cells)", fontsize=9)
        for a in (a0, a1):
            a.set_axis_off()
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.5, label="P(flood)")
    fig.suptitle(title, y=1.0, fontsize=12)
    plt.show()

if len(va_days_s):
    map_grid(vP, vY, va_days_s, f"VALIDATION — {len(va_days_s)} day(s)")
else:
    print("no validation split — plotting test days only")
map_grid(P,  Y,  te_days_s, f"TEST — {len(te_days_s)} day(s)")

## 9 · Confusion map per test day (TP / FP / FN / TN)

Each land cell coloured by outcome at the best-F1 threshold (`THRESH`). POD = TP/(TP+FN), FAR = FP/(TP+FP).

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from sklearn.metrics import precision_recall_curve

yl, pl = Y[:, land_mask].ravel(), P[:, land_mask].ravel()
prec, rec, thr = precision_recall_curve(yl, pl)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
THRESH = float(thr[max(0, f1[:-1].argmax())])           # <- set manually to explore
print(f"threshold = best-F1 = {THRESH:.3f}  (pooled test F1 {f1[:-1].max():.3f})")

CMAP = ListedColormap(["#e8e8e8", "#ff7f00", "#1f78b4", "#33a02c"])   # TN, FP, FN, TP
CMAP.set_bad("white")
NORM = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], CMAP.N)
LEG = [Patch(facecolor="#33a02c", label="TP (hit)"),
       Patch(facecolor="#1f78b4", label="FN (miss)"),
       Patch(facecolor="#ff7f00", label="FP (false alarm)"),
       Patch(facecolor="#e8e8e8", label="TN")]

n = len(te_days_s); ncol = 3; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.4 * nrow), squeeze=False)
for i, d in enumerate(te_days_s):
    pred = P[i] >= THRESH; y = Y[i].astype(bool)
    cat = np.full((GRID_R, GRID_C), np.nan)
    cat[land_mask & ~y & ~pred] = 0      # TN
    cat[land_mask & ~y &  pred] = 1      # FP
    cat[land_mask &  y & ~pred] = 2      # FN
    cat[land_mask &  y &  pred] = 3      # TP
    tp = int((cat == 3).sum()); fn = int((cat == 2).sum()); fp = int((cat == 1).sum())
    pod = tp / (tp + fn) if tp + fn else float("nan")
    far = fp / (tp + fp) if tp + fp else float("nan")
    ax = axes[i // ncol][i % ncol]
    ax.imshow(np.ma.masked_invalid(cat), cmap=CMAP, norm=NORM, interpolation="none")
    ax.set_title(f"{str(d)}   TP {tp}  FN {fn}  FP {fp}\nPOD {pod:.2f}  FAR {far:.2f}",
                 fontsize=9)
    ax.set_axis_off()
for j in range(n, nrow * ncol):
    axes[j // ncol][j % ncol].set_axis_off()
fig.legend(handles=LEG, loc="lower center", ncol=4, frameon=False)
fig.suptitle(f"Test-day confusion maps  (threshold {THRESH:.3f})", y=1.0)
plt.tight_layout(rect=[0, 0.04, 1, 1]); plt.show()

## 10 · Add GLM lightning — retrain (glm=True) & evaluate on the same split

Re-runs training with `--glm` on the **identical manifest split** (read from `manifest.parquet`, so train/test are exactly as §7), for the **same number of epochs as the images-only baseline**, saving to `ddp_results_glm.npz` (the baseline `ddp_results.npz` is untouched). Then the same PR-AUC + confusion-map evaluation, side by side.

In [ ]:
import os, subprocess, numpy as np
RESULTS_NPZ = CACHE_DIR / "ddp_results.npz"          # images-only baseline (from §7)
GLM_RESULTS = CACHE_DIR / "ddp_results_glm.npz"
n_ep = int(np.load(RESULTS_NPZ, allow_pickle=True)["hist"][:, 0].max())   # match baseline epochs
cmd = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2",
       "convlstm_june_train.py", "--epochs", str(n_ep), "--batch", str(BATCH_PER_GPU),
       "--workers", "0", "--glm", "--tag", "glm"]
env = {**os.environ, "NCCL_P2P_DISABLE": "1", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
print("launching GLM run (epochs", n_ep, "):", " ".join(cmd[1:]), flush=True)
proc = subprocess.Popen(cmd, cwd=str(ROOT / "signatures/notebooks"), env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    if any(k in line for k in ("[ddp]", "TEST", "Error", "Traceback", "Killed")):
        print(line.rstrip())
proc.wait()
assert proc.returncode == 0, "GLM training failed — see log above"

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

def _summ(npz):
    r = np.load(npz, allow_pickle=True); P, Y = r["probs"], r["trues"]
    yl, pl = Y[:, land_mask].ravel(), P[:, land_mask].ravel()
    return r, P, Y, [str(d) for d in r["te_days"]], average_precision_score(yl, pl), \
        roc_auc_score(yl, pl), yl.mean()

rb, Pb, Yb, _, apb, rocb, base = _summ(RESULTS_NPZ)
rg, Pg, Yg, days_g, apg, rocg, _ = _summ(GLM_RESULTS)
print(f"base rate {base*100:.2f}%   (same {len(days_g)}-day test split)")
print(f"images-only : TEST PR-AUC {apb:.3f} ({apb/base:.1f}x)  ROC {rocb:.3f}")
print(f"GOES + GLM  : TEST PR-AUC {apg:.3f} ({apg/base:.1f}x)  ROC {rocg:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.3))
ax.plot(rb["hist"][:, 0], rb["hist"][:, 2], "o-", ms=3, label="images-only")
ax.plot(rg["hist"][:, 0], rg["hist"][:, 2], "s-", ms=3, label="GOES + GLM")
ax.axhline(base, color="k", ls="--", lw=0.8)
ax.set_xlabel("epoch"); ax.set_ylabel("test PR-AUC"); ax.legend()
ax.set_title("test PR-AUC per epoch — images-only vs GOES+GLM")
plt.tight_layout(); plt.show()

In [ ]:
# same confusion maps as §9, for the GOES+GLM model
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from sklearn.metrics import precision_recall_curve

yl, pl = Yg[:, land_mask].ravel(), Pg[:, land_mask].ravel()
prec, rec, thr = precision_recall_curve(yl, pl)
f1 = 2 * prec * rec / (prec + rec + 1e-9)
THRESH = float(thr[max(0, f1[:-1].argmax())])
print(f"GOES+GLM threshold = best-F1 = {THRESH:.3f}  (pooled test F1 {f1[:-1].max():.3f})")

CMAP = ListedColormap(["#e8e8e8", "#ff7f00", "#1f78b4", "#33a02c"]); CMAP.set_bad("white")
NORM = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], CMAP.N)
LEG = [Patch(facecolor="#33a02c", label="TP (hit)"), Patch(facecolor="#1f78b4", label="FN (miss)"),
       Patch(facecolor="#ff7f00", label="FP (false alarm)"), Patch(facecolor="#e8e8e8", label="TN")]
n = len(days_g); ncol = 3; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.4 * nrow), squeeze=False)
for i, d in enumerate(days_g):
    pred = Pg[i] >= THRESH; y = Yg[i].astype(bool)
    cat = np.full((GRID_R, GRID_C), np.nan)
    cat[land_mask & ~y & ~pred] = 0; cat[land_mask & ~y & pred] = 1
    cat[land_mask & y & ~pred] = 2;  cat[land_mask & y & pred] = 3
    tp = int((cat == 3).sum()); fn = int((cat == 2).sum()); fp = int((cat == 1).sum())
    pod = tp / (tp + fn) if tp + fn else float("nan"); far = fp / (tp + fp) if tp + fp else float("nan")
    ax = axes[i // ncol][i % ncol]
    ax.imshow(np.ma.masked_invalid(cat), cmap=CMAP, norm=NORM, interpolation="none")
    ax.set_title(f"{d}   TP {tp}  FN {fn}  FP {fp}\nPOD {pod:.2f}  FAR {far:.2f}", fontsize=9)
    ax.set_axis_off()
for j in range(n, nrow * ncol):
    axes[j // ncol][j % ncol].set_axis_off()
fig.legend(handles=LEG, loc="lower center", ncol=4, frameon=False)
fig.suptitle(f"GOES+GLM test-day confusion maps  (threshold {THRESH:.3f})", y=1.0)
plt.tight_layout(rect=[0, 0.04, 1, 1]); plt.show()

## Notes
- **Tiny data:** ~21 CST days (16 train / 5 test, random, no val) — a wiring check, not converged. More days is the real lever.
- **Same-day = diagnostic**, not a forecast. For D-1, shift the frame window in §2.
- CLI: `torchrun --nproc_per_node=2 convlstm_june_train.py --epochs 25 --batch 3`.